In [1]:
import sqlite3
import pandas as pd

# Connect to Chinook database
conn = sqlite3.connect('chinook.db')

print("Connected to Chinook database")

Connected to Chinook database


In [2]:
# Configure pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 30)

In [3]:
SCHEMA REFERENCE
Key Tables andColumns
tracks

TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer
Milliseconds, Bytes, UnitPrice

albums

AlbumId, Title, ArtistId

artists

ArtistId, Name

customers

CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId

invoices

InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total

invoice_items

InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity

genres

GenreId, Name

media_types

MediaTypeId, Name

employees

EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email

SyntaxError: invalid syntax (1543158634.py, line 1)

In [4]:
"""
What CASE does
- CASE statements allow conditional logic in SQL - like if/else in programming.
  Essential for:
    - Creating categorical features (for ML)
    - Binning continious data
    - Conditional aggregation
    - Data cleaning/transformation


Two Types of CASE

Type 1. Simple CASE (Comparing one column to values)

CASE column_name
    WHEN value1 THEN result1
    WHEN value2 THEN result2
    ELSE default_result
END


Type 2: Searched CASE (any condition)

CASE 
    WHEN condition1 THEN result1
    WHEN condition2 THEN result2
    ELSE default_result
END

"""

'\nWhat CASE does\n- CASE statements allow conditional logic in SQL - like if/else in programming.\n  Essential for:\n    - Creating categorical features (for ML)\n    - Binning continious data\n    - Conditional aggregation\n    - Data cleaning/transformation\n\n\nTwo Types of CASE\n\nType 1. Simple CASE (Comparing one column to values)\n\nCASE column_name\n    WHEN value1 THEN result1\n\n\n'

In [6]:
"""
Example 1: Customer Segmentation by Spending

Problem Context: We want to identify high-value customers to target fir
marketing campaigns. Instead of looking at raw spending numbers, we need
clear categories.

Logical Approach:
1. Calculate total spending per customer (aggregate)
2. Define meaningful thresholds (45+ =high, 40+ =medium, rest =low)
3. Assign each customer to a tier
4. Use this for business decisions (VIP programs, email campaigns, etc.)

"""




query = """

        -- STEP 1: Join tables to connect customers with their invoices
        -- STEP 2: Group by customer to aggregate their spending
        -- STEP 3: Use CASE to categorize based on their total spending

        SELECT
            Customer.FirstName,
            SUM(Invoice.Total) AS TotalSpending, -- Aggregate: sum all invoices per customer

            -- CASE evaluates conditions top-to-bottom, first TRUE wins
            CASE
                -- Most specific condition first (highest threshold)
                WHEN SUM(Invoice.Total) >= 45 THEN 'High Value'

                -- Medium threshold (only reached if >= 45 failed)
                WHEN SUM(Invoice.Total) >= 40 THEN 'Medium Value'

                -- ELSE catches everything else (spending < 40)
                ELSE 'Low Value'
            END AS CustomerTier

        FROM Customer
        JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
        GROUP BY Customer.CustomerId, Customer.FirstName -- One row per customer
        ORDER BY TotalSpending DESC
        LIMIT 20;



        -- KEY INSIGHT: ORDER matters in CASE!
        -- If we put "WHEN >= 40" first, customers with 45+ would be labeled "Medium"
        -- because the first TRUE condition wins and stops evaluation


        -- BUSINESS USE: Now we can:
        -- - Send VIP offers to "High Value" customers
        -- - Target "Medium Value" with upsell campaigns
        -- - Re-engage "Low Value" with discounts
        
        
        """

result = pd.read_sql_query(query, conn)
print(result)





    FirstName  TotalSpending  CustomerTier
0      Helena          49.62    High Value
1     Richard          47.62    High Value
2        Luis          46.62    High Value
3    Ladislav          45.62    High Value
4        Hugh          45.62    High Value
5       Frank          43.62  Medium Value
6       Julia          43.62  Medium Value
7        Fynn          43.62  Medium Value
8      Astrid          42.62  Medium Value
9      Victor          42.62  Medium Value
10      Terhi          41.62  Medium Value
11  František          40.62  Medium Value
12   Isabelle          40.62  Medium Value
13   Johannes          40.62  Medium Value
14       Luís          39.62     Low Value
15   François          39.62     Low Value
16      Bjørn          39.62     Low Value
17       Jack          39.62     Low Value
18        Dan          39.62     Low Value
19    Heather          39.62     Low Value


In [7]:
"""
Example 2: Conditional Aggregation (COUNT with conditions)

Problem Context: We need to understand genre pricing strategies. Are some genres
dominated by cheap tracks while others have more premium content? This helps with inventory
and pricig decisions.


Logical Approach:
1. We can't just COUNT - we need conditional counting
2. Trick: CASE returns 1(true) or 0(false), then SUM counts the 1s
3. This creates "pivot-like" analysis without complex PIVOT syntax
4. Calculate percentage to see proportions clearly

"""



query = """
            -- GOAL: For each genre, count cheap vs expensive tracks seperately
            -- CHALLENGE: COUNT doesn't accept WHERE conditions per column
            -- SOLUTION: USe CASE to return 1 or 0, then SUM to count

            SELECT 
                Genre.Name as Genre,
                COUNT(*) AS TotalTracks,  -- Total tracks (baseline)


                -- Conditional counting pattern: SUM(CASE WHEN condition THEN 1 ELSE 0 END)
                -- This counts how many tracks match the condition

                SUM(CASE
                    WHEN Track.UnitPrice <= 0.99 THEN 1  -- Return 1 if cheap
                    ELSE 0                               -- Return 0 if not
                END) AS CheapTracks,

                SUM(CASE
                    WHEN Track.UnitPrice > 0.99 THEN 1   -- Return 1 if expensive
                    ELSE 0                               -- Return 0 if not
                END) AS ExpensiveTracks,

                -- Calculate percentage of expensive tracks
                -- Multiply by 100.0 (not 100) to force decimal division
                ROUND(
                    100.0 * SUM(CASE WHEN Track.UnitPrice > 0.99 THEN 1 ELSE 0 END) / COUNT(*),
                    2
                ) AS PctExpensive

            FROM Track
            JOIN Genre ON Track.GenreId = Genre.GenreId
            GROUP BY Genre.Name
            ORDER BY PctExpensive DESC;





            -- WHY THIS WORKS:
            -- For each track, CASE evaluates and returns 1 or 0
            -- SUM adds up all the 1s = count of matching tracks
            -- Example for a genre with 100 tracks (80 cheap, 20 expensive):
            -- CheapTracks = SUM(1,1,1......0,0,0) = 80
            -- ExpensiveTracks = SUM(0,0,0,0......1,1,1) = 20
            -- PctExpensive = (20 / 100) * 100 = 20%


            -- BUSINESS INSIGHT: Genres with high PctExpensive might be:
            -- - Premium Content (classical, TV shows)
            -- - Opportunities for price optimization
            -- - Different customer segments
            
        
        """

result = pd.read_sql_query(query, conn)
print(result)

                 Genre  TotalTracks  CheapTracks  ExpensiveTracks  \
0             TV Shows           93            0               93   
1      Science Fiction           13            0               13   
2     Sci Fi & Fantasy           26            0               26   
3                Drama           64            0               64   
4               Comedy           17            0               17   
5                World           28           28                0   
6           Soundtrack           43           43                0   
7        Rock And Roll           12           12                0   
8                 Rock         1297         1297                0   
9               Reggae           58           58                0   
10            R&B/Soul           61           61                0   
11                 Pop           48           48                0   
12               Opera            1            1                0   
13               Metal          37

In [9]:
"""
Example 3: Data Cleaning - Fixing NULL or Invalid Values

Problem Context: Real data is messy. Before ML or analysis, we need to handle:
- Missing Values (NULL)
- Impossible values (0 length tracks, nagative prices)
- Out-of-range values (1 second track = data error)


Logical Approach
1. Identify problematic values (NULL, 0, unrealistic values)
2. Define reasonable defaults or corrections
3. Apply business rules (minimum track length, standard price)
4. Create cleaned columns alongside original (for comparison/audit)

"""

query = """
        -- DATA QUALITY PROBLEM: Some tracks have NULl or invalid values
        -- SOLUTION: Use CASE to detect and fix issues


        SELECT
            Track.Name,
            Track.Milliseconds,  -- Original value (might be NULL or invalid)

            -- Clean track length: handle NULL and unralistic values
            CASE
              -- FIRST: Check for NULL (most critical)
              WHEN Track.Milliseconds IS NULL THEN 0   -- Default to 0 or could use average

              -- SECOND: Check for unrealistically short tracks (<  10 seconds)
              -- This catches data entry errors or corrupted data
              WHEN Track.Milliseconds < 10000 THEN 10000 -- Set minimum realistic length

              -- ELSE: Value is valid, keep it
              ELSE Track.Milliseconds
            END AS CleanedLength,

            -- Clean price: handle NULL and zero prices
            CASE
                -- NULL price: missing data, use standard price
                WHEN Track.UnitPrice IS NULL THEN 0.99

                -- Zero price = data error (music isn't free in this store)
                WHEN Track.UnitPrice = 0 THEN 0.99

                -- Valid price: keep original
                ELSE Track.UnitPrice
            END AS CleanedPrice

        FROM Track
        LIMIT 20;





        -- REASONING BEHIND DEFAULTS:
        -- - 0 for NULL length: Clearly flags as "unknown" without breaking calculations
        -- - 10000ms (10 sec) minimum: Reasonable lower bound for actual music tracks
        -- - 0.99 default price: Standard price in the Chinook database


        -- ALTERNATIVE APPROACHES:
        -- - Could use AVG(Milliseconds) instead of 0 or NULL
        -- - Could flag as "needs review" instead of auto-fixing
        -- - Could delete invalid records entirely

        -- FOR ML: Clean data is essential
        -- - Models can't handle NULL in most cases
        -- - Unrealistic values create outliers that skew training
        -- - Document your cleaning rules for reproducibility
        
            
        
        """

result = pd.read_sql_query(query, conn)
print(result)



                             Name  Milliseconds  CleanedLength  CleanedPrice
0   For Those About To Rock (W...        343719         343719          0.99
1               Balls to the Wall        342562         342562          0.99
2                 Fast As a Shark        230619         230619          0.99
3               Restless and Wild        252051         252051          0.99
4            Princess of the Dawn        375418         375418          0.99
5           Put The Finger On You        205662         205662          0.99
6                 Let's Get It Up        233926         233926          0.99
7                Inject The Venom        210834         210834          0.99
8                      Snowballed        203102         203102          0.99
9                      Evil Walks        263497         263497          0.99
10                         C.O.D.        199836         199836          0.99
11             Breaking The Rules        263288         263288          0.99